<a href="https://colab.research.google.com/github/AdvayRoongta/audio-initial-dataset/blob/main/scripts/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inference: run the conflict dataset through Qwen2-Audio

Feeds every example in `examples.json` to Qwen2-Audio-7B-Instruct and saves the
raw generations to a new file. That is all it does -- no scoring, no accuracy,
no TDR. Those come later, computed *from* the file this produces.

**Self-contained.** The download cells below are the same ones as
`load_datasets.ipynb`, so a fresh Colab session runs this notebook top to bottom
without touching any other notebook. Clips already cached in Drive are detected
and skipped, so re-runs cost nothing.

Needs a GPU. On an A100 the whole sweep takes a few minutes.

## Setup

In [ ]:
import os

# Land at the repo root regardless of how this notebook was opened:
#  - fresh Colab (via the badge above): only this notebook exists yet -> clone
#  - local Jupyter opened from inside scripts/ -> just cd up one level
#  - local Jupyter opened from the repo root -> already there, no-op
if os.path.exists("requirements.txt"):
    pass
elif os.path.exists("../requirements.txt"):
    %cd ..
else:
    # The repo is private, so an anonymous clone dies with "could not read
    # Username for 'https://github.com'". Create a fine-grained PAT with
    # Contents: read-only, add it as a Colab secret named GH_TOKEN (key icon
    # in the left sidebar), and turn on notebook access for this notebook.
    import subprocess

    from google.colab import userdata

    token = userdata.get("GH_TOKEN").strip()
    url = "https://x-access-token:" + token + "@github.com/AdvayRoongta/audio-initial-dataset.git"
    r = subprocess.run(["git", "clone", "--depth", "1", url, "repo"],
                       capture_output=True, text=True)
    if r.returncode:
        # git echoes the whole URL in its errors, so redact before showing it.
        raise RuntimeError(r.stderr.replace(token, "***"))
    %cd repo

!pip install -q -r requirements.txt


## Where the data lives

On Colab the clips go to **your own Google Drive** so they persist across
sessions, and so does the predictions file this notebook writes at the end --
Colab wipes `/content` (including the cloned repo) when the runtime recycles.

In [ ]:
import os
from pathlib import Path

USE_DRIVE = True      # on Colab, cache the clips in your Google Drive
FORCE_RELOAD = False  # True = re-download even if a cached copy exists

REPO_ROOT = Path.cwd()

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")  # one-time authorisation prompt per session
    DATA_ROOT = Path("/content/drive/MyDrive/audio-initial-dataset")
else:
    DATA_ROOT = REPO_ROOT

DATA_DIR = DATA_ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Manifests and examples.json store paths like "data/raw/<cat>/audio/<clip>.wav",
# i.e. relative to DATA_ROOT -- so they stay valid whether the clips live in
# Drive or in the repo. Any notebook that reads them resolves DATA_ROOT / path.
# The symlink below is a convenience so plain relative paths also work when the
# working directory is the repo root. We never touch a real existing data/ dir.
repo_data = REPO_ROOT / "data"
if DATA_ROOT != REPO_ROOT and not repo_data.exists():
    os.symlink(DATA_ROOT / "data", repo_data)
    print(f"symlinked {repo_data} -> {DATA_ROOT / 'data'}")
elif DATA_ROOT != REPO_ROOT and not repo_data.is_symlink():
    print(f"note: {repo_data} already exists, leaving it alone "
          f"(resolve clips with DATA_ROOT / audio_path instead)")

print("DATA_ROOT:", DATA_ROOT)
print("clips ->  ", DATA_DIR)


## Source datasets: one per conflict category

In [ ]:
import json

import soundfile as sf
from datasets import load_dataset

# DATA_ROOT / DATA_DIR / FORCE_RELOAD come from the storage cell above.
N_ROWS = 100  # examples.json references clip index 99 -- do NOT lower this
              # (bump it later for the real dataset build)

# One public, ungated Hugging Face dataset per conflict category / person.
# Each is loadable with plain `datasets.load_dataset(hf_id)` -- no
# trust_remote_code and no HF login required, so everyone can just run this.
CATEGORIES = {
    "emotional_prosody": {          # Advay: cheerful text vs. angry delivery, etc.
        "hf_id": "confit/cremad-parquet",   # CREMA-D: acted emotional speech
        "hf_split": "train",
        "primary_label_field": "emotion",
    },
    # On hold for the paper for now -- initial results showed ESC-50 clips
    # weren't a good fit for this category. Left here, commented out, so
    # it's easy to re-enable: just uncomment and re-run this cell.
    # "non_speech_event": {            # Ankur: calm text vs. sirens/screams/etc. in audio
    #     "hf_id": "renumics/esc50",          # ESC-50: 50-class environmental sounds
    #     "hf_split": "train",
    #     "primary_label_field": "label",
    # },
    "speaker_identity": {            # Danity: text claims one speaker, voice is another
        "hf_id": "xbgoose/ravdess",         # RAVDESS: acted speech, known actor gender
        "hf_split": "train",
        "primary_label_field": "gender",
    },
    "sarcasm_pacing": {              # Vishnu: sincere text vs. sarcastic delivery
        "hf_id": "macabdul9/SarcasmDetection_Mustard",  # MUStARD: TV-show sarcasm clips
        "hf_split": "test",              # this dataset only ships a "test" split
        "primary_label_field": "label",
    },
}


## Download the clips

Identical to `load_datasets.ipynb`. `is_cached()` short-circuits when a previous
run already wrote the clips + manifest, so this is a no-op after the first time.

In [ ]:
def label_to_str(value, feature):
    """Some HF label columns are ints backed by a ClassLabel; others are plain strings."""
    if hasattr(feature, "int2str"):
        return feature.int2str(value)
    return str(value)


def is_cached(name: str, n_rows: int) -> bool:
    """True if a previous run already wrote this category's clips + manifest."""
    manifest_path = DATA_DIR / name / "manifest.json"
    if not manifest_path.exists():
        return False
    try:
        manifest = json.loads(manifest_path.read_text())
    except json.JSONDecodeError:
        return False
    if len(manifest) < n_rows:
        return False  # cached fewer rows than we want now -> re-pull
    return all((DATA_ROOT / row["audio_path"]).exists() for row in manifest)


def load_category(name: str, cfg: dict, n_rows: int) -> None:
    print(f"\n=== {name} ({cfg['hf_id']}) ===")

    if is_cached(name, n_rows) and not FORCE_RELOAD:
        print(f"cached -> {DATA_DIR / name}  (skipping download)")
        return

    ds = load_dataset(cfg["hf_id"], split=cfg["hf_split"])
    if n_rows < len(ds):
        ds = ds.select(range(n_rows))

    out_dir = DATA_DIR / name
    audio_dir = out_dir / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)

    features = ds.features
    label_field = cfg["primary_label_field"]

    manifest = []
    for i, example in enumerate(ds):
        audio = example["audio"]  # {"array": np.ndarray, "sampling_rate": int, ...}
        clip_id = f"{name}_{i:03d}"
        audio_path = audio_dir / f"{clip_id}.wav"
        sf.write(audio_path, audio["array"], audio["sampling_rate"])

        primary_label = label_to_str(example[label_field], features[label_field])

        # Keep every other column (minus the raw audio array, which we already
        # wrote to disk) as free-form metadata so nothing from the source
        # dataset gets lost even if we didn't explicitly plan for a field.
        source_metadata = {k: v for k, v in example.items() if k != "audio"}

        manifest.append({
            "clip_id": clip_id,
            "audio_path": str(audio_path.relative_to(DATA_ROOT)),
            "duration_sec": round(len(audio["array"]) / audio["sampling_rate"], 3),
            "primary_label": primary_label,
            "source_metadata": source_metadata,
        })

    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
    print(f"saved {len(manifest)} clips -> {audio_dir}")
    print(f"wrote manifest  -> {manifest_path}")


for name, cfg in CATEGORIES.items():
    load_category(name, cfg, N_ROWS)


## The conflict examples

`examples.json` is the hand-written dataset: each entry pairs a clip with a
`text_prompt` that asserts something the audio contradicts.

In [ ]:
import json

with open("examples.json") as f:
    examples = json.load(f)

# Paths in examples.json are relative to DATA_ROOT.
missing = [e["id"] for e in examples if not (DATA_ROOT / e["audio_path"]).exists()]
print(f"{len(examples)} examples, {len(missing)} missing audio")
if missing:
    # The download cell above should have fetched these -- check that it ran.
    print("missing ->", missing[:5])


## Processor

`encode()` is copied from `experiments.ipynb` unchanged, deliberately: the
prompt the model answers here is then exactly the prompt the attention and
knockout analysis look at.

In [ ]:
from transformers import AutoProcessor

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
SR = processor.feature_extractor.sampling_rate

print(f"sampling rate {SR}")


In [ ]:
import inspect

import librosa

# transformers renamed this kwarg (audios= -> audio=) and silently ignores the
# wrong one, so pick whichever this version actually takes.
AUDIO_KWARG = "audio" if "audio" in inspect.signature(processor.__call__).parameters else "audios"


def encode(example):
    """Run one example through the processor. Returns inputs + duration."""
    path = DATA_ROOT / example["audio_path"]
    audio, _ = librosa.load(path, sr=SR)

    conversation = [{"role": "user", "content": [
        {"type": "audio", "audio_url": str(path)},
        {"type": "text", "text": example["text_prompt"]},
    ]}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text], sampling_rate=SR, return_tensors="pt",
                       **{AUDIO_KWARG: [audio]})
    return inputs, len(audio) / SR


## Load the model

`sdpa`, not `eager`: `experiments.ipynb` forces eager only because its attention
sections need the weights. Generation doesn't, and sdpa is faster.

On an A100 this loads in bf16 -- same memory as fp16, wider exponent range. The
4-bit branch is kept so the notebook still runs on a smaller GPU, but note that
4-bit perturbs activations, so record which path was taken (the run header
below does).

In [ ]:
import torch
from transformers import Qwen2AudioForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError(
        f"torch cannot see a GPU (torch {torch.__version__}, "
        f"cuda {torch.version.cuda}).\n"
        "  - build ends in '+cpu' / cuda is None -> this is a CPU-only build of "
        "torch; reinstall the CUDA build\n"
        "  - on Colab -> Runtime > Change runtime type > GPU\n"
        "  - on a GPU server -> check `nvidia-smi` works and the driver is loaded")

load_kwargs = dict(device_map="auto", attn_implementation="sdpa")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb < 20:
    # 8.4B params in fp16 = 16.8 GB, which won't fit a 15 GB T4.
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    QUANTIZATION = "4bit"
    print(f"{vram_gb:.0f} GB VRAM -> loading in 4-bit (needs: pip install bitsandbytes)")
else:
    load_kwargs["torch_dtype"] = torch.bfloat16
    QUANTIZATION = "none"
    print(f"{vram_gb:.0f} GB VRAM -> loading in bfloat16")

model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print("loaded:", model.dtype, model.device)


## Run inference

Greedy decoding (`do_sample=False`) so the run is reproducible. Batch size 1 --
61 examples on an A100 is a couple of minutes.

Saves to Drive every 5 examples, not just at the end, so a dropped Colab
session doesn't cost the whole run.

`raw_output` is stored exactly as decoded -- no strip, no lowercasing, no
mapping to a label. `truncated` flags a generation that hit the token cap
without finishing, which otherwise looks identical to a short answer.

In [ ]:
import hashlib
import importlib.metadata
import subprocess
import time
from datetime import datetime, timezone

MAX_NEW_TOKENS = 64
SAVE_EVERY = 5

OUT_DIR = DATA_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "qwen2audio_predictions.json"

_eos = model.generation_config.eos_token_id
EOS_IDS = set(_eos) if isinstance(_eos, (list, tuple)) else {_eos}


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


def _git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                              text=True).stdout.strip() or None
    except Exception:
        return None


# Everything needed to say exactly what produced these numbers. Built once, up
# front, so the incremental saves carry it too.
RUN = {
    "model_id": MODEL_ID,
    # The HF repo can move under a tag; this is the commit actually downloaded.
    "model_revision": getattr(model.config, "_commit_hash", None),
    "quantization": QUANTIZATION,
    "dtype": str(model.dtype),
    "gpu": torch.cuda.get_device_name(0),
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    # Pins the eval set + every prompt in it: examples.json holds text_prompt.
    "examples_json_sha256": hashlib.sha256(Path("examples.json").read_bytes()).hexdigest(),
    "n_examples_in_file": len(examples),
    "repo_commit": _git_commit(),
    "packages": {p: _version(p) for p in
                 ("torch", "transformers", "bitsandbytes", "accelerate", "datasets")},
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}


def save(records, skipped):
    payload = {"run": RUN, "skipped": skipped, "predictions": records}
    OUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False))


records, skipped = [], []
t0 = time.time()

for i, e in enumerate(examples, 1):
    if not (DATA_ROOT / e["audio_path"]).exists():
        skipped.append(e["id"])
        print(f"[{i:>3}/{len(examples)}] {e['id']:<16} SKIPPED (no audio)")
        continue

    inputs, _ = encode(e)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

    new_tokens = ids[0, prompt_len:]
    raw_output = processor.batch_decode(new_tokens.unsqueeze(0), skip_special_tokens=True)[0]

    # Stopped because it ran out of budget rather than because it finished.
    n_new = int(new_tokens.shape[0])
    truncated = n_new >= MAX_NEW_TOKENS and int(new_tokens[-1]) not in EOS_IDS

    records.append({
        "id": e["id"],
        "conflict_type": e["conflict_type"],
        "audio_path": e["audio_path"],
        "text_prompt": e["text_prompt"],
        "ground_truth_audio": e["ground_truth_audio"],
        "ground_truth_text": e["ground_truth_text"],
        "raw_output": raw_output,
        "truncated": truncated,
    })

    flag = "  <-- TRUNCATED" if truncated else ""
    preview = raw_output.replace("\n", " ")[:60]
    print(f"[{i:>3}/{len(examples)}] {e['id']:<16} {preview!r}{flag}")

    if len(records) % SAVE_EVERY == 0:
        save(records, skipped)
        print(f"{'':>6}  saved {len(records)} records -> {OUT_PATH}")

save(records, skipped)
print(f"\n{len(records)} generated, {len(skipped)} skipped, {time.time() - t0:.0f}s")
print("wrote", OUT_PATH)
